# Notebook exploring example training dataset

In [ ]:
from pathlib import Path

import numpy as np
import rasterio
import xarray as xr
from ipywidgets import widgets
from rasterio.control import GroundControlPoint
from rasterio.crs import CRS

xr.set_options(
    display_style="html",
    display_expand_data_vars=True,
    display_expand_attrs=True,
    display_expand_coords=True,
    display_expand_data=True,
)

## Read in example Ready-to-train and raw file

In [ ]:
data_toppath = Path(
    "/Users/gordonlogie/Documents/Projects/Internal_Projects/Prescient/arctic_showcase/data/autoice_dataset"
)
rrt_path = data_toppath.joinpath("readytotrain", "21762830")
raw_path = data_toppath.joinpath("raw", "21762848")

data_outpath = data_toppath.joinpath("data_exploration_outputs")
data_outpath.mkdir(parents=True, exist_ok=True)

In [ ]:
rrt_files = list(rrt_path.glob("*_prep.nc"))
rtt_label_files = list(rrt_path.glob("*_reference.nc"))
raw_files = list(raw_path.glob("*.nc"))

## Select image to explore

In [ ]:
im_select = widgets.Dropdown(
    options=[(f.stem, f) for f in sorted(rrt_files)], description="Select RTT file:"
)

display(im_select)

In [ ]:
test_rtt_file = im_select.value
test_rtt_date = test_rtt_file.stem.split("_")[0]
test_raw_file = [f for f in raw_files if test_rtt_date in f.stem][0]
test_label_file = [f for f in rtt_label_files if test_rtt_date in f.stem][0]

select_fname = test_rtt_file.stem
select_outpath = data_outpath.joinpath(f"{select_fname}")

## Examine each dataset

In [ ]:
raw_ds = xr.open_dataset(test_raw_file)
rtt_ds = xr.open_dataset(test_rtt_file)
rtt_label_ds = xr.open_dataset(test_label_file)

In [ ]:
print(f"Raw Dataset: {test_raw_file.name}")
print("CRS Information:", raw_ds.crs if hasattr(raw_ds, "crs") else "no crs information available")
print("Number of variables:", len(raw_ds.data_vars))
var_list_raw = list(raw_ds.data_vars.keys())
raw_ds

In [ ]:
print(f"Ready-to-Train Dataset: {test_rtt_file.name}")
print("CRS Information:", rtt_ds.crs if hasattr(rtt_ds, "crs") else "no crs information available")
print("Number of variables:", len(rtt_ds.data_vars))
var_list_rtt = list(rtt_ds.data_vars.keys())
rtt_ds

In [ ]:
print("Ready-to-Train Label Dataset:", test_label_file.name)
print("CRS Information:", rtt_ds.crs if hasattr(rtt_ds, "crs") else "no crs information available")
print("Number of variables:", len(rtt_ds.data_vars))
var_list_rtt_label = list(rtt_label_ds.data_vars.keys())
rtt_label_ds

In [ ]:
missing_in_rtt = set(var_list_raw) - set(var_list_rtt)
print("Variables in raw dataset but missing in ready-to-train dataset:")
for var in missing_in_rtt:
    print(f"- {var}")

missing_in_raw = set(var_list_rtt) - set(var_list_raw)
print("\nVariables in ready-to-train dataset but missing in raw dataset:")
for var in missing_in_raw:
    print(f"- {var}")

## Georeference and output data variables
This section allows a user to output a select variable from either the raw or RTT dataset as a georeferenced geotiff. 

In [ ]:
SAR_DIMS = {"sar_lines", "sar_samples"}
TWO_KM_DIMS = {"2km_grid_lines", "2km_grid_samples"}


def project_and_output_variable_as_geotiff(select_fname, dataset_name, dataset, var, out_path):
    da = dataset[var]

    if set(da.dims) not in (SAR_DIMS, TWO_KM_DIMS):
        print(f"Skipping {var}: no georeferencing available for dims {da.dims}")
        return

    data = da.values
    is_2km = set(da.dims) == TWO_KM_DIMS

    if "sar_grid_line" in dataset.data_vars:
        lines = dataset["sar_grid_line"].values
        samples = dataset["sar_grid_sample"].values
        lats = dataset["sar_grid_latitude"].values
        lons = dataset["sar_grid_longitude"].values
        if is_2km:
            scale_l = data.shape[0] / dataset.sizes["sar_lines"]
            scale_s = data.shape[1] / dataset.sizes["sar_samples"]
            gcps = [
                GroundControlPoint(
                    row=lines[i] * scale_l, col=samples[i] * scale_s, x=lons[i], y=lats[i]
                )
                for i in range(len(lines))
            ]
        else:
            gcps = [
                GroundControlPoint(row=lines[i], col=samples[i], x=lons[i], y=lats[i])
                for i in range(len(lines))
            ]
    else:
        lats = dataset["sar_grid2d_latitude"].values
        lons = dataset["sar_grid2d_longitude"].values
        n_dim0, n_dim1 = lats.shape
        # First array dim is lines (rows), second is samples (cols) — naming is misleading
        line_pos = np.linspace(0, data.shape[0] - 1, n_dim0)
        sample_pos = np.linspace(0, data.shape[1] - 1, n_dim1)
        gcps = [
            GroundControlPoint(row=line_pos[i], col=sample_pos[j], x=lons[i, j], y=lats[i, j])
            for i in range(n_dim0)
            for j in range(n_dim1)
        ]

    # Bypass EPSG lookup (broken proj.db from conda) — use PROJ string directly
    wgs84 = CRS.from_string("+proj=longlat +datum=WGS84 +no_defs")

    out_path = out_path.joinpath(dataset_name)
    out_path.mkdir(parents=True, exist_ok=True)
    out_file = out_path / f"{select_fname}_{dataset_name}_{var}_georeferenced.tif"

    with rasterio.open(
        out_file,
        "w",
        driver="GTiff",
        height=data.shape[0],
        width=data.shape[1],
        count=1,
        dtype=data.dtype,
        crs=None,
    ) as dst:
        dst.write(data, 1)
        dst.gcps = (gcps, wgs84)

    print(f"Written to {out_file}")
    print(f"GCPs embedded: {len(gcps)}")

### Add lat and lon from RTT to RTT Labels

In [ ]:
rtt_label_ds = rtt_label_ds.assign(
    {
        "sar_grid2d_latitude": rtt_ds["sar_grid2d_latitude"],
        "sar_grid2d_longitude": rtt_ds["sar_grid2d_longitude"],
    }
)

### Select Dataset to output
RTT labels are appended to RTT dataset to output

In [ ]:
datasets = {
    "Raw Dataset": raw_ds,
    "Ready-to-Train Dataset": rtt_ds,
    "Ready-to-Train Label Dataset": rtt_label_ds,
}

ds_select = widgets.Dropdown(
    options=list(datasets.keys()),
    description="Select Dataset:",
)

variable_select = widgets.SelectMultiple(
    options=list(datasets[ds_select.value].data_vars),
    description="Select Variable(s):",
    value=list(datasets[ds_select.value].data_vars),
)


def update_variables(change):
    variable_select.options = list(datasets[change["new"]].data_vars)
    variable_select.value = list(datasets[change["new"]].data_vars)


ds_select.observe(update_variables, names="value")

widgets.VBox([ds_select, variable_select])

In [ ]:
select_vars = variable_select.value
select_ds_name = ds_select.value

for select_var in select_vars:
    project_and_output_variable_as_geotiff(
        select_fname=select_fname,
        dataset_name=select_ds_name.replace(" ", "_"),
        dataset=datasets[select_ds_name],
        var=select_var,
        out_path=select_outpath,
    )